# 01 - Data preparation

This notebook reads and checks the five MSCI workbooks. It then writes the three local CSV files used by the other notebooks. Raw and derived MSCI data are not committed; see `DATA_NOTICE.md`.

## Input files and checks

Install the project in editable mode, then run this notebook from the repository root or from `notebooks/`. All input checks run before a processed file is written.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from robust_dm_factor_allocation import load_config, validate_returns

In [ ]:
working_directory = Path.cwd().resolve()
repository_root = (
    working_directory.parent
    if working_directory.name == "notebooks"
    else working_directory
)
config_path = repository_root / "config" / "config.yaml"
metadata_path = repository_root / "data" / "metadata" / "index_metadata.csv"
raw_directory = repository_root / "data" / "raw"
processed_directory = repository_root / "data" / "processed"

if not config_path.is_file() or not metadata_path.is_file():
    raise FileNotFoundError(
        "Repository files were not found. Start Jupyter in the repository root "
        "or in its notebooks directory."
    )

config = load_config(config_path)
factor_columns = list(config.factor_columns)
benchmark = config.benchmark
series_order = [*factor_columns, benchmark]

In [ ]:
metadata_date_columns = [
    "launch_date",
    "first_observation",
    "last_observation",
    "download_date",
]
index_metadata = pd.read_csv(metadata_path, parse_dates=metadata_date_columns)
required_metadata_columns = {
    "series_id",
    "launch_date",
    "first_observation",
    "last_observation",
    "raw_filename",
}
missing_metadata_columns = required_metadata_columns.difference(index_metadata.columns)
if missing_metadata_columns:
    raise ValueError(
        f"Metadata is missing columns: {sorted(missing_metadata_columns)}"
    )
if not index_metadata["series_id"].is_unique:
    raise ValueError("Metadata contains duplicate series_id values.")

metadata_by_series = index_metadata.set_index("series_id")
missing_series = [name for name in series_order if name not in metadata_by_series.index]
if missing_series:
    raise ValueError(f"Metadata is missing configured series: {missing_series}")

required_files = {
    series_id: raw_directory / metadata_by_series.at[series_id, "raw_filename"]
    for series_id in series_order
}
missing_files = [path.name for path in required_files.values() if not path.is_file()]
if missing_files:
    formatted_names = "\n  - ".join(missing_files)
    raise FileNotFoundError(
        "Licensed source files are not part of this repository. Obtain them "
        "independently, confirm that your use is permitted, and place them in "
        f"data/raw with the documented filenames:\n  - {formatted_names}\n"
        "See data/raw/README.md and DATA_NOTICE.md."
    )

## Read monthly index levels

The parser expects MSCI's two-column monthly export. It rejects gaps and non-positive levels, then assigns each observation to its calendar month-end.

In [ ]:
def read_monthly_index_level(file_path, series_id, header_row=6, sheet_name=0):
    raw = pd.read_excel(
        file_path,
        sheet_name=sheet_name,
        header=header_row,
        engine="xlrd",
    ).dropna(axis="columns", how="all")
    if raw.shape[1] < 2:
        raise ValueError(f"Expected date and index-level columns in {file_path.name}.")

    observations = raw.iloc[:, :2].copy()
    observations.columns = ["date", "index_level"]
    observations["date"] = pd.to_datetime(observations["date"], errors="coerce")
    observations["index_level"] = pd.to_numeric(
        observations["index_level"]
        .astype("string")
        .str.strip()
        .str.replace(",", "", regex=False),
        errors="coerce",
    )
    observations = observations.dropna(subset=["date", "index_level"])
    if observations.empty:
        raise ValueError(f"No valid observations found in {file_path.name}.")
    if (observations["index_level"] <= 0).any() or not np.isfinite(
        observations["index_level"].to_numpy()
    ).all():
        raise ValueError(f"Index levels must be finite and positive in {file_path.name}.")

    observations["month"] = observations["date"].dt.to_period("M")
    observations = observations.sort_values("date", kind="stable")
    monthly = observations.groupby("month", sort=True).tail(1)
    observed_months = pd.PeriodIndex(monthly["month"])
    expected_months = pd.period_range(observed_months.min(), observed_months.max(), freq="M")
    missing_months = expected_months.difference(observed_months)
    if len(missing_months):
        raise ValueError(
            f"Missing months in {file_path.name}: "
            f"{missing_months.astype(str).tolist()}"
        )

    series = monthly.set_index("month")["index_level"].sort_index()
    series.index = series.index.to_timestamp("M")
    series.index.name = "date"
    series.name = series_id
    return series

In [ ]:
series_by_id = {
    series_id: read_monthly_index_level(required_files[series_id], series_id)
    for series_id in series_order
}

for series_id, series in series_by_id.items():
    expected_first = metadata_by_series.at[series_id, "first_observation"].to_period("M")
    expected_last = metadata_by_series.at[series_id, "last_observation"].to_period("M")
    observed_months = series.index.to_period("M")
    if observed_months.min() != expected_first or observed_months.max() != expected_last:
        raise ValueError(
            f"Observed date range for {series_id} does not match index_metadata.csv."
        )

monthly_index_levels = pd.concat(
    [series_by_id[name] for name in series_order],
    axis="columns",
    join="outer",
    sort=False,
).sort_index()
monthly_index_levels.index.name = "date"

## Returns and shared sample

Returns are simple month-over-month percentage changes. The portfolio notebooks only keep months available for every sleeve and the benchmark. Missing values are not filled.

In [ ]:
monthly_returns_full = monthly_index_levels.pct_change(fill_method=None)
monthly_returns_common = validate_returns(
    monthly_returns_full.dropna(how="any"),
    factor_columns=factor_columns,
    market_column=benchmark,
    missing=config.missing_policy,
)

observed_levels = monthly_index_levels.to_numpy(dtype=float, na_value=np.nan)
observed_levels = observed_levels[np.isfinite(observed_levels)]
observed_full_returns = monthly_returns_full.to_numpy(dtype=float, na_value=np.nan)
observed_full_returns = observed_full_returns[np.isfinite(observed_full_returns)]
if not (observed_levels > 0).all():
    raise ValueError("All observed index levels must be positive.")
if (observed_full_returns <= -1).any():
    raise ValueError("Monthly returns must be greater than -100%.")

## Write local files

The next cell writes three CSV files to the ignored `data/processed/` directory. Notebook outputs and MSCI results also stay local.

In [ ]:
processed_directory.mkdir(parents=True, exist_ok=True)
monthly_index_levels.to_csv(processed_directory / "monthly_index_levels.csv")
monthly_returns_full.to_csv(processed_directory / "monthly_returns_full.csv")
monthly_returns_common.to_csv(processed_directory / "monthly_returns_common.csv")

## Before continuing

If a date, layout or continuity check fails, fix the source file or metadata before continuing.